# 全 A 股多因子选股：一键复现

**目标**：把「数据 → 因子 → 模型 → 验证 → 清单」这条链在一个 notebook 里跑通。

两种用法：

- **快速版（默认，30 秒）**：读仓库里已有的产物（面板统计、CV 结果、清单），适合交作业时演示。
- **完整重跑版**：把各单元里被注释掉的 `!python scripts/...` 打开，按顺序执行。重建面板约 40 分钟，训练约 25 分钟。

> 所有数字都来自 `reports/` 下冻结的报告，没有在这里重新调参。

## 0. 环境与数据自检

In [ ]:
import sys, os, json
import numpy as np, pandas as pd
ROOT = r'D:/quant'
print(sys.version.split()[0], '| numpy', np.__version__, '| pandas', pd.__version__)
for p in ['data/big/arr.npz','data/big/daily_ret.npz','checkpoints/lgbm_big.txt',
          'reports/purged_cv_summary.json','live/screen_2026-10-30.csv']:
    print('%-42s %s' % (p, '✓' if os.path.exists(f'{ROOT}/{p}') else '✗ 缺失'))

## 1. 面板：规模、日期、因子

In [ ]:
z = np.load(f'{ROOT}/data/big/arr.npz', allow_pickle=True)
X, Y2, M = z['X'], z['Y2'], z['M']
dates = pd.to_datetime(pd.Series(z['dates']).astype(str))
feats = list(z['feats'])
print('X %s | 交易日 %d | 股票 %d | 因子 %d' % (X.shape, X.shape[0], X.shape[1], X.shape[2]))
print('区间 %s ~ %s | 封存段起点 %s' % (dates.iloc[0].date(), dates.iloc[-1].date(), z['lock_start']))
print('因子：', ', '.join(feats))

## 2. 因子：单因子 IC 与排名

口径：日频截面 rank IC vs `Y2`（t 日信号 → t+1 收盘买入 → 持有 5 日，截面已去均值）。
完整打分卡（含中性化、十分位、换手）见 `reports/factor_scorecard.csv`。

In [ ]:
from scipy.stats import spearmanr
# 只算最近 60 个交易日，滚动展示（全量在 factor_scorecard.py）
idx = np.where(M.any(1))[0][-60:]
rows = []
for j, f in enumerate(feats):
    ics = []
    for t in idx:
        m = M[t] & np.isfinite(X[t, :, j]) & np.isfinite(Y2[t])
        if m.sum() > 300:
            ics.append(spearmanr(X[t, m, j], Y2[t, m]).statistic)
    s = pd.Series(ics).dropna()
    rows.append((f, s.mean(), s.mean()/s.std() if s.std() else np.nan, len(s)))
ic = pd.DataFrame(rows, columns=['factor','IC','ICIR','n']).sort_values('IC', key=abs, ascending=False)
ic.head(10).round(4)

## 3. 模型：LGBM + Transformer + Ridge 元模型

模型权重已冻结在 `checkpoints/`，这里只读取元信息，不重训。
**重要**：`preds.npz` 里的 `va` 与 `te` 是同一批样本（2025H1），不是两次独立验证——
真正的样本外只有 5 折 Purged CV 和已消耗的封存段。

In [ ]:
meta = json.load(open(f'{ROOT}/reports/big_models_meta.json'))
lm = json.load(open(f'{ROOT}/checkpoints/lgbm_meta.json'))
print('LGBM best_iter =', lm['best_iter'])
print('Stacking coef =', np.round(meta['coef'], 4), '(LGBM / TF / 因子合成 / 截距)')
print('线性权重非零的因子：', {k: round(v,3) for k,v in meta['weights'].items() if v > 0})

## 4. 验证：5 折 Purged CV

In [ ]:
cv = json.load(open(f'{ROOT}/reports/purged_cv_summary.json'))
print(json.dumps(cv, ensure_ascii=False, indent=1)[:800])

## 5. 落地：当期禁买清单

`is_d1 == 1` 的 500 只就是「别买」的名单。注意：6 天前的面板（`ok` 过滤要求 5 日前向收益，
所以面板最后一天天然落后最新行情 6 个交易日），实盘前需刷新。

In [ ]:
sc = pd.read_csv(f'{ROOT}/live/screen_2026-10-30.csv', dtype={'symbol': str})
print('打分日 %s | %d 只 | D1 %d 只' % (sc.date.iloc[0], len(sc), sc.is_d1.sum()))
sc.nsmallest(10, 'score')[['symbol','name','industry','score','pct']]

## 6. 完整重跑（可选，耗时长）

按顺序执行；每一步都会写 `logs/`（谁、什么参数、跑多久、成没成功）。

```python
# !python scripts/big_download_hfq.py        # 后复权日线（断点续传）
# !python scripts/big_features.py            # 重建面板，约 40 分钟
# !python scripts/big_train.py               # LGBM + TF×3 + Ridge，约 25 分钟
# !python scripts/purged_cv.py               # 5 折 CV，约 10 分钟
# !python scripts/live_screen.py             # 当期清单
```

**不要**跑 `lockbox_eval.py`：封存段已一次性消耗（`reports/lockbox_state.json` `runs=1`），
重跑等于自欺欺人。

## 7. 结论（这段是报告里要写的）

1. 模型能稳定预测 5 日收益的**排序**（5 折 CV IC 0.1041 ± 0.0094，五折全正），
2. 但预测力集中在**垫底那一撮**（D1 5 日 −0.99%，D10 +0.52%），A 股不能做空全市场，
3. 所以唯一可落地形态是**禁买清单**：年化贡献 3~5%，且逐年衰减（+5.85% → +4.67% → +2.87%），
4. 八条替代路径（SVM、多空、行业动量、基本面加成、Muon、TAA、策略轮动、业绩预告事件）全部证伪。

详见 `reports/big_report.md`、`reports/text_bench.md`、`reports/return_label.md`。